# CellFate-Rx — demo

A calibrated, OOD-aware multi-task model for **safe cellular rejuvenation triage**: given a cell's state `X` and a perturbation `U`, it predicts calibrated P(identity preserved / loss / death), a ΔAge with honest uncertainty, and a safety-dominant **Rejuvenation Efficacy Score (RES)**.

This notebook (1) runs the whole pipeline end-to-end on the built-in synthetic source, and (2) loads the **real Fleischer aging clock** (fit from GSE113957) and shows it predicting transcriptomic age.

> Synthetic data is deliberately trivial — linear baselines are near-perfect, so `beats_all_baselines` is expected to be **False** here (the gate doing its job). The point is to show the machinery runs; real results need a real perturbation/aging dataset.

## Setup
Clone the repo (brings the code **and** the fitted clock in `configs/clocks/`) and install.

In [ ]:
# EDIT this to your repo, then run.
REPO = "https://github.com/hagai-tyty/Cellular-Outcomes-of-Perturbations.git"

import os
if not os.path.isdir("cellfate-rx"):
    !git clone -q $REPO
%cd cellfate-rx
!pip install -q -e ".[model]"
print("installed")

## 1. Build a small dataset (synthetic source, resumable ETL)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from cellfate.data import DataConfig, QCConfig, SyntheticSource, run as build_run

ROOT = "demo_run"
build_run(
    DataConfig(out=ROOT, gene_panel=f"{ROOT}/panel.json", n_genes=64,
               qc=QCConfig(min_genes=5, max_mito_frac=0.5), label_tau=0.5,
               clock="random",  # synthetic smoke: ages are placeholders (use a fitted clock for real runs)
               split_fracs=(0.6, 0.2, 0.1, 0.1),
               split_regimes=("scaffold", "cell_line", "both"), primary_regime="cell_line", seed=0),
    sources=[SyntheticSource(name="synth", n_lines=5, n_compounds=21,
                             n_cells_per_condition=4, n_filler_genes=50,
                             n_scaffold_families=7, seed=1)])
print("dataset built ->", ROOT)

## 2. Train a small calibrated ensemble

In [ ]:
from cellfate.training.train_model import TrainConfig, run as train_run
train_run(TrainConfig(dataset_dir=ROOT, out=ROOT, regime="cell_line",
                      d_cell=48, d_u=48, latent_dim=48, p_drop=0.2, lr=3e-3,
                      epochs=10, patience=4, batch_size=256, ensemble_size=3,
                      base_seed=0, conformal_levels=(0.90,), device="cpu"))
print("bundle trained (members + temperature + conformal + OOD)")

## 3. Score cells → safety probs + ΔAge interval + OOD + RES

In [ ]:
import glob
from cellfate.inference import Predictor, score_shard

pred = Predictor(ROOT)
shard = sorted(glob.glob(f"{ROOT}/shards/*.parquet"))[0]
responses, cell_ids = score_shard(pred, shard)
for resp, cid in list(zip(responses, cell_ids))[:5]:
    r = resp.model_dump()
    print(f"{cid:20s} P_safe={r['p_identity_preserved']:.2f} "
          f"ΔAge={r['delta_age_mean']:+.2f} {r['delta_age_interval']} "
          f"OOD={not r['in_distribution']}  RES={r['rejuvenation_efficacy_score']:.2f}  {r['status']}")

## 4. Evaluate — baselines + acceptance gates (per regime)

In [ ]:
import json
from cellfate.evaluation.evaluate_cli import EvalConfig, evaluate
gates = evaluate(EvalConfig(bundle=ROOT, dataset=ROOT, regimes=("cell_line", "scaffold"),
                            out=f"{ROOT}/reports"))
print(json.dumps(gates, indent=2, default=str))

## 5. The real Fleischer aging clock
Loaded from `configs/clocks/fleischer_clock.json` (fit on GSE113957 human dermal fibroblasts, ages 1–96). It consumes the **full transcriptomic profile** (not the model's HVG panel) and returns a transcriptomic age. Top-weighted genes are classic fibroblast/aging markers (COL1A1, FN1, VIM).

In [ ]:
import numpy as np
from cellfate.data.aging import LinearClock

clock = LinearClock.from_json("configs/clocks/fleischer_clock.json")
print(f"clock: {len(clock.weights)} symbol-keyed genes; top markers:", list(clock.weights)[:6])

# toy: a batch of fibroblast-like profiles over the clock's genes -> predicted ages
genes = list(clock.weights)[:1000]
rng = np.random.default_rng(0)
expr = rng.gamma(2.0, 1.0, size=(8, 1000))          # 8 pseudo-cells
ages = clock.predict_age(expr, genes)
print("predicted transcriptomic ages:", np.round(ages, 1))
print("(varies across cells -> the clock reads the profile; on symbol-mismatched genes it would return only the intercept)")